# Caso 3 · Cox07 | Severidad, prima pura y supervivencia
Cartera de seguros de automóvil. Adaptación del [caso 3 de A. M. Mayoral](https://asunmayoral.github.io/GLM/caso3/caso3_continuas_positivas.html).

🏵 Tres preguntas: ¿cuánto cuesta un parte?, ¿cuánto cuesta un año de cobertura?, ¿cuándo ocurre un evento?

La unidad de observación cambia: parte → póliza-año → intervalo de riesgo. No confundimos coste positivo con coste agregado, ni tiempo hasta un parte con tiempo hasta la baja.

El cuaderno es exploratorio y reproducible. Las conclusiones numéricas se obtienen al ejecutarlo; no se presuponen.

In [ ]:
SEMILLA_CURSO <- 20252026L
set.seed(SEMILLA_CURSO)
options(repr.plot.width=11, repr.plot.height=6)
paquetes <- c("dplyr","ggplot2","readr","glmmTMB","DHARMa","survival","MASS")
faltan <- setdiff(paquetes, rownames(installed.packages()))
if(length(faltan)) install.packages(faltan, repos="https://cloud.r-project.org")
stopifnot(all(vapply(paquetes, requireNamespace, logical(1), quietly=TRUE)))
library(dplyr)
library(ggplot2)
library(survival)
theme_set(theme_minimal(base_size=14))
base_equipo <- "https://raw.githubusercontent.com/asunmayoral/GLM/b05aeaecde9dca0049ea0ca55fbe162a9b964cbc/caso2/datos_equipos_auto/equipo_07/"
huellas <- c("cartera.csv"="6a8a1cd5a22ec0813be10b4d2173b28f","panel.csv"="813f20b73d15d78bb5777a21268d07c8","siniestros.csv"="9523e2bd2665c492d8dc1fbe5502d7de","cartera.rds"="949486c1f1c5b51ef43f441748e1932d")
archivo_equipo <- function(nombre) {
  ruta <- file.path("datos", nombre)
  if(!file.exists(ruta)) {
    ruta <- file.path(tempdir(), nombre)
    if(!file.exists(ruta)) download.file(paste0(base_equipo,nombre),ruta,mode="wb",quiet=TRUE)
  }
  stopifnot(unname(tools::md5sum(ruta)) == huellas[[nombre]])
  ruta
}
cartera_original <- readr::read_csv(archivo_equipo("cartera.csv"),show_col_types=FALSE)
panel <- readr::read_csv(archivo_equipo("panel.csv"),show_col_types=FALSE)
siniestros <- readr::read_csv(archivo_equipo("siniestros.csv"),show_col_types=FALSE)
cartera_rds <- readRDS(archivo_equipo("cartera.rds"))
stopifnot(!anyDuplicated(cartera_original$id_poliza),sum(cartera_original$n_siniestros)==nrow(siniestros),sum(panel$n_partes_anual)==nrow(siniestros))
set.seed(SEMILLA_CURSO)
ids_ajuste <- sample(cartera_original$id_poliza,floor(.75*nrow(cartera_original)))
ids_reserva <- setdiff(cartera_original$id_poliza,ids_ajuste)
str(cartera_rds,max.level=1)

## 0 · Construir las unidades de análisis
La exposición se mide en años con expo_anual. El coste de un parte está en euros. Los predictores son características conocidas al inicio; excluimos coste, conteos futuros y bonus-malus actual. Los costes cero se conservan en el panel, no en Gamma. La reserva se separa por póliza para evitar que un mismo cliente aparezca en ajuste y validación.

In [ ]:
rasgos <- c("id_poliza","id_agencia","id_region","edad_conductor","antiguedad_vehiculo","tipo_vehiculo","zona_circulacion","uso","km_anuales","sexo")
clientes <- cartera_original |> select(all_of(rasgos)) |> mutate(across(c(id_agencia,id_region,tipo_vehiculo,zona_circulacion,uso,sexo),factor), edad10=edad_conductor/10, veh10=antiguedad_vehiculo/10, km10=km_anuales/10000)
stopifnot(all(is.finite(siniestros$coste_k)),all(siniestros$coste_k>=0),!anyDuplicated(panel[c("id_poliza","anio")]))
partes <- left_join(siniestros,clientes,by="id_poliza")
anual <- panel |> left_join(siniestros |> group_by(id_poliza,anio) |> summarise(coste=sum(coste_k),.groups="drop"),by=c("id_poliza","anio")) |> mutate(coste=coalesce(coste,0)) |> left_join(clientes,by="id_poliza")
stopifnot(abs(sum(anual$coste)-sum(partes$coste_k))<.01,all(anual$expo_anual>=0),all(anual$coste[anual$expo_anual==0]==0))
vars <- c("edad10","veh10","tipo_vehiculo","zona_circulacion","uso","km10","sexo")
stopifnot(all(complete.cases(clientes[vars])))
positivos <- filter(partes,coste_k>0)
print(c(partes=nrow(partes),ceros=sum(partes$coste_k==0),polizas=nrow(clientes),anualidades=nrow(anual)))
A <- filter(positivos,id_poliza %in% ids_ajuste)
V <- filter(positivos,id_poliza %in% ids_reserva)
P <- filter(anual,expo_anual>0,id_poliza %in% ids_ajuste)
Q <- filter(anual,expo_anual>0,id_poliza %in% ids_reserva)
f <- reformulate(vars,"coste_k")
g <- reformulate(vars,"coste")
ggplot(positivos,aes(coste_k))+geom_histogram(bins=50,fill="gold",colour="white")+labs(x="Coste por parte (€)",y="Partes")
ggplot(positivos,aes(tipo_vehiculo,coste_k))+geom_violin(fill="gold",alpha=.6)+geom_boxplot(width=.12)+scale_y_log10()+labs(y="Coste (€), escala log")

## 3.1 · Gamma: severidad
🔰 E[Y|x] = μ; Var(Y|x) = φ μ². Con enlace log, μ = exp(η), η = Xβ. Un incremento unitario de un predictor multiplica la media por exp(β). Edad10 representa diez años. El enlace inverso es otra parametrización y puede producir predictores no admisibles. Comparamos con gaussiano y Gamma inverso, sin confundir significación con capacidad predictiva.

In [ ]:
m0 <- glm(f,data=A,family=gaussian())
mG <- glm(f,data=A,family=Gamma("log"))
mI <- try(glm(f,data=A,family=Gamma("inverse"),start=c(1/mean(A$coste_k),rep(0,length(coef(mG))-1))),silent=TRUE)
summary(mG)
exp(cbind(razon=coef(mG),confint.default(mG)))
drop1(mG,test="F")
AIC(m0,mG)
if(inherits(mI,"glm") && mI$converged) print(AIC(mG,mI))
diagnostico <- function(m) {
  plot(fitted(m),residuals(m,type="pearson"),xlab="Media ajustada",ylab="Residuo Pearson",pch=16,col="#77777755")
  abline(h=0,col="tomato")
}
diagnostico(mG)
pr <- predict(mG,newdata=V,type="link",se.fit=TRUE)
head(data.frame(media=exp(pr$fit),inf=exp(pr$fit-1.96*pr$se.fit),sup=exp(pr$fit+1.96*pr$se.fit)))

## 3.2 · Familia, escala y dispersión
La varianza crece con μ: normal (p=0), Gamma (p=2), inversa gaussiana (p=3). El modelo lognormal ajusta log(Y), no Y. El factor de Duan corrige la retransfomación de la media bajo error logarítmico homogéneo. Para comparar AIC en euros se añade 2 Σ log(Y) al AIC lognormal. La pendiente de la nube media-varianza es descriptiva, no una prueba independiente.

In [ ]:
mIG <- glm(f,data=A,family=inverse.gaussian("log"))
mLN <- lm(update(f,log(coste_k)~.),data=A)
duan <- mean(exp(residuals(mLN)))
comparacion <- data.frame(modelo=c("Gamma","Inversa gaussiana","Lognormal"),AIC=c(AIC(mG),AIC(mIG),AIC(mLN)+2*sum(log(A$coste_k))))
print(comparacion)
preds <- list(Gamma=predict(mG,V,type="response"),IG=predict(mIG,V,type="response"),LN=exp(predict(mLN,V))*duan)
metricas <- function(y,p) c(MAE=mean(abs(y-p)),RMSE=sqrt(mean((y-p)^2)),sesgo=mean(p-y),observado=sum(y),predicho=sum(p))
print(sapply(preds,function(p) metricas(V$coste_k,p)))
nube <- A |> mutate(grupo=ntile(fitted(mG),10)) |> group_by(grupo) |> summarise(mu=mean(coste_k),var=var(coste_k),.groups="drop")
ggplot(nube,aes(mu,var))+geom_point(size=3,colour="tomato")+geom_smooth(method="lm",se=FALSE)+scale_x_log10()+scale_y_log10()
A$rp2 <- pmax(residuals(mG,type="pearson")^2,1e-10)
disp <- glm(reformulate(vars,"rp2"),data=A,family=Gamma("log"))
summary(disp)
mD <- glmmTMB::glmmTMB(f,data=A,family=Gamma("log"),dispformula=~tipo_vehiculo+veh10)
summary(mD)

## 3.3 · Gamma mixto: ¿hay agencias caras?
Añadimos efectos aleatorios de agencia y póliza. El intercepto mide heterogeneidad persistente. La pendiente aleatoria de antigüedad del vehículo por agencia es una hipótesis que debemos contrastar, no una obligación. Un efecto de agencia se encoge hacia cero cuando hay poca información. En enlace log, fijar efectos aleatorios a cero no equivale a integrar su distribución: para un intercepto normal de varianza σ², la media marginal incluye exp(σ²/2). No interpretamos el ICC gaussiano como un ICC exacto Gamma.

In [ ]:
mMix0 <- glmmTMB::glmmTMB(f,data=A,family=Gamma("log"))
mMix <- glmmTMB::glmmTMB(update(f,.~.+(1|id_agencia)+(1|id_poliza)),data=A,family=Gamma("log"))
mSlope <- try(glmmTMB::glmmTMB(update(f,.~.+(1+veh10|id_agencia)+(1|id_poliza)),data=A,family=Gamma("log")),silent=TRUE)
valido <- function(m) inherits(m,"glmmTMB") && isTRUE(m$sdr$pdHess) && isTRUE(m$fit$convergence==0)
print(glmmTMB::VarCorr(mMix))
print(AIC(mMix0,mMix))
if(valido(mSlope)) print(AIC(mMix,mSlope)) else message("Pendiente aleatoria no identificada o no convergente")
if(valido(mMix)) {
  print(head(glmmTMB::ranef(mMix)$cond$id_agencia))
  sim <- DHARMa::simulateResiduals(mMix,n=250)
  plot(sim)
  print(DHARMa::testDispersion(sim))
  print(head(predict(mMix,V,type="response",re.form=NA,allow.new.levels=TRUE)))
}

## 3.4 · Tweedie: prima pura
Sea S la suma de N costes. Si N es Poisson y los costes son Gamma independientes bajo el modelo, S tiene masa en cero y parte continua positiva. E[S] = E[N] E[Y]. Ajustamos la tasa anual R=S/e con pesos e: Var(R|x)=φ μ^p/e. Esto respeta la agregación temporal; un offset sin pesos en el coste total no impone esa misma varianza. Comparamos Tweedie, frecuencia Poisson × severidad Gamma y NB × Gamma. Los tres predicen euros por póliza-año. Si existen partes de coste cero, la severidad del producto incluye su probabilidad.

In [ ]:
P$tasa <- P$coste/P$expo_anual
Q$tasa <- Q$coste/Q$expo_anual
ft <- reformulate(vars,"tasa")
mT <- glmmTMB::glmmTMB(ft,data=P,weights=expo_anual,family=glmmTMB::tweedie(link="log"))
mTM <- glmmTMB::glmmTMB(update(ft,.~.+(1|id_agencia)+(1|id_poliza)),data=P,weights=expo_anual,family=glmmTMB::tweedie(link="log"))
print(glmmTMB::family_params(mT))
print(AIC(mT,mTM))
fn <- reformulate(c(vars,"offset(log(expo_anual))"),"n_partes_anual")
mF <- glm(fn,data=P,family=poisson())
mNB <- MASS::glm.nb(fn,data=P)
probpos <- mean(partes$coste_k[partes$id_poliza %in% ids_ajuste]>0)
sev <- predict(mG,Q,type="response")*probpos
Q$Tweedie <- predict(mT,Q,type="response")*Q$expo_anual
Q$PoissonGamma <- predict(mF,Q,type="response")*sev
Q$NBGamma <- predict(mNB,Q,type="response")*sev
print(sapply(Q[c("Tweedie","PoissonGamma","NBGamma")],function(p) metricas(Q$coste,p)))
if(valido(mTM)) {
  simT <- DHARMa::simulateResiduals(mTM,n=250)
  plot(simT)
  print(DHARMa::testZeroInflation(simT))
}
cal <- Q |> mutate(grupo=ntile(Tweedie/expo_anual,10)) |> group_by(grupo) |> summarise(obs=sum(coste)/sum(expo_anual),pred=sum(Tweedie)/sum(expo_anual),.groups="drop")
ggplot(cal,aes(pred,obs))+geom_abline(slope=1,intercept=0)+geom_point(colour="tomato",size=3)+labs(x="Prima prevista",y="Prima observada")

### Validación agrupada y presupuesto
Cinco particiones por póliza dentro del conjunto de ajuste. La reserva permanece intacta. El presupuesto de renovación se ilustra con un año completo y perfiles constantes; no representa una predicción de bajas, inflación ni cambios de composición. El intervalo de confianza de la media no es un intervalo de pérdidas futuras.

In [ ]:
set.seed(SEMILLA_CURSO)
fold <- setNames(sample(rep(1:5,length.out=length(ids_ajuste))),ids_ajuste)
cv <- lapply(1:5,function(k){
  test <- fold[as.character(P$id_poliza)]==k
  m <- glmmTMB::glmmTMB(ft,data=P[!test,],weights=expo_anual,family=glmmTMB::tweedie(link="log"))
  p <- predict(m,P[test,],type="response")*P$expo_anual[test]
  c(fold=k,metricas(P$coste[test],p))
})
print(do.call(rbind,cv))
renovar <- clientes
renovar$expo_anual <- 1
X <- model.matrix(delete.response(terms(ft)),renovar)
b <- glmmTMB::fixef(mT)$cond
X <- X[,names(b),drop=FALSE]
mu <- exp(drop(X%*%b))
grad <- colSums(X*mu)
se <- sqrt(drop(t(grad)%*%vcov(mT)$cond%*%grad))
print(c(presupuesto=sum(mu),IC_inf=max(0,sum(mu)-1.96*se),IC_sup=sum(mu)+1.96*se))

## 3.5 · AFT: tiempo hasta el primer parte y hasta la baja
La censura derecha significa que el evento no se observó durante el seguimiento. Su contribución a la verosimilitud es S(t), no f(t). Kaplan–Meier estima supervivencia sin imponer distribución. En AFT, exp(β) es un factor de tiempo: valores superiores a uno retrasan el evento. Analizamos ambos desenlaces por separado; la baja puede competir con el primer parte, por lo que la supervivencia del primer parte no equivale a incidencia acumulada en presencia de riesgos competitivos.

In [ ]:
T <- cartera_original |> select(id_poliza,t_primer_sin,evento,t_baja,baja,antiguedad) |> left_join(clientes,by="id_poliza")
stopifnot(all(T$evento %in% 0:1),all(T$baja %in% 0:1),all(T$t_primer_sin>0),all(T$t_baja>0))
TA <- filter(T,id_poliza %in% ids_ajuste)
TV <- filter(T,id_poliza %in% ids_reserva)
aft <- function(t,e) {
  formula <- reformulate(vars,paste0("Surv(",t,",",e,")"))
  modelos <- lapply(c("exponential","weibull","lognormal","loglogistic"),function(d) survreg(formula,data=TA,dist=d))
  names(modelos) <- c("exponential","weibull","lognormal","loglogistic")
  print(sapply(modelos,AIC))
  mejor <- modelos[[which.min(sapply(modelos,AIC))]]
  print(exp(cbind(factor_tiempo=coef(mejor),confint.default(mejor))))
  mejor
}
plot(survfit(Surv(t_primer_sin,evento)~tipo_vehiculo,data=TA),col=1:nlevels(TA$tipo_vehiculo),xlab="Años",ylab="S(t)")
legend("topright",legend=levels(TA$tipo_vehiculo),col=1:nlevels(TA$tipo_vehiculo),lty=1)
print(survdiff(Surv(t_primer_sin,evento)~tipo_vehiculo,data=TA))
aft_parte <- aft("t_primer_sin","evento")
plot(survfit(Surv(t_baja,baja)~1,data=TA),xlab="Años",ylab="Probabilidad de seguir sin baja")
aft_baja <- aft("t_baja","baja")
head(predict(aft_parte,newdata=TV,type="quantile",p=.5))

## 3.6 · Cox: riesgo instantáneo
h(t|x)=h₀(t) exp(Xβ). El hazard ratio es exp(β); no es una razón de probabilidades acumuladas. Cox estima β mediante verosimilitud parcial y no exige una forma paramétrica para h₀. Schoenfeld contrasta proporcionalidad. La fragilidad de agencia permite dependencia; el efecto aleatorio no sustituye la comprobación de proporcionalidad.
Para recurrencias usamos t_k, no el número anual de partes como fecha. Los intervalos terminan en cada parte o al final de cobertura. El historial se calcula antes del evento.

In [ ]:
fc <- reformulate(vars,"Surv(t_primer_sin,evento)")
cox <- coxph(fc,data=TA,x=TRUE)
summary(cox)
print(cox.zph(cox))
plot(cox.zph(cox))
cox_baja <- coxph(reformulate(vars,"Surv(t_baja,baja)"),data=TA,x=TRUE)
print(cox.zph(cox_baja))
cox_ag <- coxph(update(fc,.~.+frailty(id_agencia)),data=TA)
summary(cox_ag)
print(concordance(Surv(t_primer_sin,evento)~predict(cox,newdata=TV),data=TV,reverse=TRUE))
# Andersen–Gill: tiempo desde alta, historial de partes previos
eventos <- siniestros |> group_by(id_poliza) |> arrange(t_k,.by_group=TRUE) |> mutate(inicio=lag(t_k,default=0),previos=row_number()-1L) |> ungroup() |> transmute(id_poliza,inicio,fin=t_k,evento=1L,previos)
colas <- eventos |> group_by(id_poliza) |> summarise(inicio=max(fin),previos=n(),.groups="drop")
fin <- T |> select(id_poliza,antiguedad) |> left_join(colas,by="id_poliza") |> transmute(id_poliza,inicio=coalesce(inicio,0),fin=antiguedad,evento=0L,previos=coalesce(previos,0L))
print(summary(fin$fin-fin$inicio))
# El tiempo de cobertura CSV está redondeado: tolerancia de 0.001 años.
stopifnot(all(fin$fin-fin$inicio>=-.001),all(eventos$fin>eventos$inicio))
AG <- bind_rows(eventos,filter(fin,fin>inicio)) |> left_join(clientes,by="id_poliza")
stopifnot(sum(AG$evento)==nrow(siniestros))
ag <- coxph(reformulate(c(vars,"previos","cluster(id_poliza)"),"Surv(inicio,fin,evento)"),data=filter(AG,id_poliza %in% ids_ajuste),x=TRUE)
summary(ag)

## 3.7 · Tres lentes sobre el mismo primer parte
Cortamos el seguimiento exacto en intervalos anuales. Poisson a trozos modela la tasa con offset log(exposición). Cox usa tiempos exactos. Complementary log-log modela la probabilidad de evento por intervalo: log(−log(1−p))=log(e)+η si la duración está fijada antes de conocer el evento. Por ello usamos solo años completos potenciales en la versión discreta y no introducimos como predictor el tiempo exacto al evento. La censura dentro del año se excluye de esa comparación discreta; no es una equivalencia exacta en esa muestra.

In [ ]:
cortes <- seq(0,ceiling(max(TA$t_primer_sin)),by=1)
pp <- survSplit(Surv(t_primer_sin,evento)~.,data=TA,cut=cortes[-c(1,length(cortes))],start="inicio",episode="tramo")
pp$duracion <- pp$t_primer_sin-pp$inicio
pp$tramo <- factor(pp$tramo)
stopifnot(all(pp$duracion>0))
mp <- glm(reformulate(c(vars,"tramo","offset(log(duracion))"),"evento"),data=pp,family=poisson())
# Ventanas discretas: evento dentro del año o año observado completo
disc <- filter(pp,evento==1 | duracion>=1-1e-8)
mc <- glm(reformulate(c(vars,"tramo"),"evento"),data=disc,family=binomial("cloglog"))
comunes <- Reduce(intersect,list(names(coef(cox)),names(coef(mp)),names(coef(mc))))
comunes <- setdiff(comunes,"(Intercept)")
print(data.frame(efecto=comunes,Cox=exp(coef(cox)[comunes]),Poisson=exp(coef(mp)[comunes]),Cloglog=exp(coef(mc)[comunes])))

## 3.8 · Estudio de caso: artículo de investigación
Propuesta de trabajo principal: **prima pura por tres caminos**. El análisis anterior también prepara distribución de costes, agencias, renovación, sexo, historial y presupuesto.

- Pregunta: ¿qué modelo estima mejor el coste anual fuera de muestra?
- Método: comparar Tweedie y frecuencia × severidad en la misma reserva, con exposición y calibración.
- Extensión propuesta: modelo de dispersión dependiente del perfil, o validación de renovación por historial previo. Justificar con bibliografía y resultados; no presentar como novedad confirmada antes de revisar el encargo.
- Resultados: completar con las tablas de 3.2–3.7, intervalos y limitaciones.
- Verdad del generador: inspeccionar el RDS y localizar los parámetros exactos antes de comparar. No confundir parámetros de frecuencia con parámetros de coste.
- Limitaciones: datos simulados, dependencia por póliza, censura, cambios de cartera y ausencia de validación temporal prospectiva.

Fuente: [material docente](https://asunmayoral.github.io/GLM/caso3/caso3_continuas_positivas.html).

In [ ]:
print(names(cartera_rds))
print(sapply(Q[c("Tweedie","PoissonGamma","NBGamma")],function(p) metricas(Q$coste,p)))
sessionInfo()